# Module 6 — SCD Type 1/2, Schema Evolution
Exam domain: **Data Modeling**

Databricks notebook version — `%sql` MERGE syntax and a widget-driven change date.

In [ ]:
dbutils.widgets.text("change_date", "2024-03-01")
change_date = dbutils.widgets.get("change_date")

In [ ]:
from pyspark.sql import functions as F
dim_customer_scd2 = spark.createDataFrame(
    [(1, "Alice", "Madrid", "2024-01-01", None, True)],
    ["customer_id", "name", "city", "effective_date", "end_date", "is_current"]
).withColumn("effective_date", F.to_date("effective_date"))

dim_customer_scd2.write.format("delta").mode("overwrite").saveAsTable("dim_customer_scd2")

## SCD Type 2 with SQL MERGE (two-statement pattern)

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW changes AS
SELECT * FROM VALUES (1, 'Alice', 'Barcelona') AS t(customer_id, name, city);

In [ ]:
spark.sql(f"""
MERGE INTO dim_customer_scd2 t
USING changes s
ON t.customer_id = s.customer_id AND t.is_current = true AND t.city <> s.city
WHEN MATCHED THEN UPDATE SET end_date = '{change_date}', is_current = false
""")

spark.sql(f"""
INSERT INTO dim_customer_scd2
SELECT s.customer_id, s.name, s.city, '{change_date}' AS effective_date, NULL AS end_date, true AS is_current
FROM changes s
JOIN dim_customer_scd2 t ON t.customer_id = s.customer_id
WHERE t.effective_date = '{change_date}' AND t.is_current = false
""")

In [ ]:
%sql
SELECT * FROM dim_customer_scd2 ORDER BY effective_date

## Schema evolution with Auto Loader / DLT
On Databricks streaming/DLT, prefer `cloudFiles.schemaEvolutionMode` (Module 4)
over manual `mergeSchema` for streaming sources — it's designed to survive
schema drift without stopping the pipeline. For batch `MERGE`/`INSERT`, use
`spark.databricks.delta.schema.autoMerge.enabled = true` at the session level,
or `.option("mergeSchema", "true")` per write, exactly as in the Colab
version.

In [ ]:
%sql
SET spark.databricks.delta.schema.autoMerge.enabled = true;